<a href="https://colab.research.google.com/github/Longcodepro/plant-disease-ml/blob/feat%2Fconvnext-onnx-rust-khoi/notebooks/04_convnext_tiny_domain_adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 Phase 2 (Domain Adaptation): Mixed-Domain Fine-Tuning (PlantVillage + PlantDoc)

- **Vấn đề thực nghiệm (Domain Shift):** Mô hình `ConvNeXt-Tiny` (`EXP-03`) đạt **99.95% Macro-F1** trên phòng lab (`PlantVillage`) nhưng tụt xuống **30.93% Top-1 / 54.24% Top-3 (28.22% Macro-F1)** trên **236 ảnh thực tế ngoài trời (`PlantDoc Test`)**.
- **Giải pháp:**
  1. Khởi tạo từ checkpoint tốt nhất `models/convnext_tiny_khoi.pth`.
  2. Trộn `PlantVillage/train` (70.295 ảnh) + `PlantDoc/train` (2.336 ảnh ngoài trời, **oversample ~10x** để chiếm ~25% mỗi batch).
  3. Bật **Field-Robust Augmentation** (`RandomPerspective`, `ColorJitter` mạnh hơn, `RandomErasing` mô phỏng che khuất lá/cành).
  4. Chấm song song trên **cả 2 tập**: `PlantVillage Valid` (Lab) và `PlantDoc Test` (Ngoài trời) sau mỗi epoch, xuất ra `models/convnext_tiny_field_v2_int8.onnx`.

In [ ]:
import os, sys, json, copy, time, importlib, subprocess
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ConcatDataset, DataLoader
from torchvision import datasets, models, transforms

IN_COLAB = "google.colab" in sys.modules
GITHUB_REPO = "https://github.com/Longcodepro/plant-disease-ml.git"
GITHUB_BRANCH = "feat/convnext-onnx-rust-khoi"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = Path("/content/plant-disease-ml")
    if not (PROJECT_DIR / ".git").exists():
        clone = subprocess.run(
            ["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, str(PROJECT_DIR)],
            capture_output=True, text=True,
        )
        if clone.returncode != 0:
            subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(PROJECT_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

    os.chdir(PROJECT_DIR)
    importlib.invalidate_caches()

    preferred_drive = Path("/content/drive/MyDrive/Mac_PhamGiaKhoi/Dự án/ML/plant-disease-ml")
    DRIVE_OUT_DIR = preferred_drive if preferred_drive.exists() else Path("/content/drive/MyDrive/plant-disease-ml-outputs")
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(PROJECT_DIR)
    DRIVE_OUT_DIR = PROJECT_DIR

OUT_MODELS_DIR = DRIVE_OUT_DIR / "models"
OUT_REPORTS_DIR = DRIVE_OUT_DIR / "notebooks" / "eda_reports"
OUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("PROJECT_DIR    :", PROJECT_DIR)
print("OUT_MODELS_DIR :", OUT_MODELS_DIR)
print("OUT_REPORTS_DIR:", OUT_REPORTS_DIR)
print("DEVICE         :", device)

In [ ]:
# 1. Chuẩn bị cả PlantVillage (Lab) và PlantDoc (In-the-Wild) trên SSD tạm /content/data
DATA_ROOT = Path("/content/data") if IN_COLAB else PROJECT_DIR / "data"

if IN_COLAB and not (DATA_ROOT / "train").exists():
    from scripts.download_data import DEFAULT_GDRIVE_FILE_ID, download_from_gdrive, extract_and_organize
    drive_archive = DRIVE_OUT_DIR / "data.tar.gz"
    if drive_archive.exists():
        extract_and_organize(drive_archive, DATA_ROOT)
    else:
        tmp_arc = Path("/content/data.tar.gz")
        download_from_gdrive(DEFAULT_GDRIVE_FILE_ID, tmp_arc)
        extract_and_organize(tmp_arc, DATA_ROOT)
        tmp_arc.unlink(missing_ok=True)

PLANTDOC_DIR = DATA_ROOT / "plantdoc"
if not (PLANTDOC_DIR / "train").exists():
    from scripts.prepare_plantdoc import prepare_plantdoc
    prepare_plantdoc(DATA_ROOT / "plantdoc_raw", PLANTDOC_DIR, PROJECT_DIR / "models" / "class_indices.json")

# Tạo bản chia sạch 0% UUID leakage ngay trên SSD Colab (~2 giây bằng symlink, không cần up lại Drive)
CLEAN_PV_DIR = DATA_ROOT / "clean_split"
if not (CLEAN_PV_DIR / "train").exists():
    from scripts.audit_uuid_leakage import audit_and_optionally_split
    audit_and_optionally_split(
        data_dir=DATA_ROOT,
        out_json=OUT_REPORTS_DIR / "uuid_leakage_audit.json",
        build_clean_split=True,
        clean_dir=CLEAN_PV_DIR,
    )

print("PlantVillage Clean (0% leakage):", CLEAN_PV_DIR)
print("PlantDoc train    :", PLANTDOC_DIR / "train")

In [ ]:
# 2. Field-Robust Data Augmentation + Mixed-Domain Dataset (Oversample PlantDoc ~10x)
from notebooks.data_pipeline import IMAGENET_MEAN, IMAGENET_STD, get_eval_transforms
from notebooks.eval_engine import evaluate_model, print_evaluation_summary

IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 6
LR = 5e-5
PLANTDOC_OVERSAMPLE = 10  # 2.336 * 10 = 23.360 ảnh (~25% mỗi batch là ảnh ngoài trời)

field_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.65, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=25),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

pv_train_ds = datasets.ImageFolder(str(CLEAN_PV_DIR / "train"), transform=field_train_transform)
pd_train_ds = datasets.ImageFolder(str(PLANTDOC_DIR / "train"), transform=field_train_transform)
# Đồng nhất class_to_idx của PlantDoc (28 lớp) về đúng 38 lớp của PlantVillage
pd_train_ds.samples = [(path, pv_train_ds.class_to_idx[pd_train_ds.classes[cls_idx]]) for path, cls_idx in pd_train_ds.samples]
pd_train_ds.targets = [s[1] for s in pd_train_ds.samples]

mixed_train_ds = ConcatDataset([pv_train_ds] + [pd_train_ds] * PLANTDOC_OVERSAMPLE)
train_loader = DataLoader(mixed_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"), drop_last=True)

pv_valid_ds = datasets.ImageFolder(str(CLEAN_PV_DIR / "valid"), transform=get_eval_transforms(IMAGE_SIZE))
pv_valid_loader = DataLoader(pv_valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

pd_test_ds = datasets.ImageFolder(str(PLANTDOC_DIR / "test"), transform=get_eval_transforms(IMAGE_SIZE))
pd_test_ds.samples = [(path, pv_train_ds.class_to_idx[pd_test_ds.classes[cls_idx]]) for path, cls_idx in pd_test_ds.samples]
pd_test_ds.targets = [s[1] for s in pd_test_ds.samples]
pd_test_loader = DataLoader(pd_test_ds, batch_size=32, shuffle=False, num_workers=2)

print(f"Mixed Train Samples: {len(mixed_train_ds):,} (PV {len(pv_train_ds):,} + PlantDoc {len(pd_train_ds):,} x {PLANTDOC_OVERSAMPLE})")
print(f"Lab Valid: {len(pv_valid_ds):,} | Field Test (PlantDoc): {len(pd_test_ds):,}")

In [ ]:
# 3. Load Checkpoint Lab (nếu có) hoặc ImageNet pretrained làm điểm xuất phát & Fine-tune Domain Adaptation
BASE_CKPT_CANDIDATES = [
    OUT_MODELS_DIR / "convnext_tiny_khoi.pth",
    PROJECT_DIR / "models" / "convnext_tiny_khoi.pth",
]
BASE_CKPT = next((p for p in BASE_CKPT_CANDIDATES if p.exists()), None)

SAVE_V2_PTH = OUT_MODELS_DIR / "convnext_tiny_field_v2.pth"
SAVE_V2_ONNX = OUT_MODELS_DIR / "convnext_tiny_field_v2.onnx"
SAVE_V2_INT8 = OUT_MODELS_DIR / "convnext_tiny_field_v2_int8.onnx"

num_classes = len(pv_train_ds.classes)
if BASE_CKPT is not None:
    print(f"📥 Nạp Lab Checkpoint: {BASE_CKPT}")
    model = models.convnext_tiny(weights=None)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    ckpt = torch.load(BASE_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
else:
    print("⚠️ Chưa có convnext_tiny_khoi.pth -> khởi tạo trực tiếp từ ConvNeXt_Tiny_Weights.DEFAULT (ImageNet)")
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_score = 0.0
for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, targets in train_loader:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)
    scheduler.step()

    lab_m = evaluate_model(model, pv_valid_loader, device=device)
    field_m = evaluate_model(model, pd_test_loader, device=device)
    # Chọn checkpoint tối ưu đồng thời cả Lab & Ngoài trời
    combined = 0.4 * lab_m["macro_f1"] + 0.6 * field_m["macro_f1"]
    print(
        f"[DA {ep}/{EPOCHS}] ({time.time()-t0:.1f}s) | Train Acc {correct/total*100:.2f}% | "
        f"Lab Acc {lab_m['accuracy']*100:.2f}% | Field (PlantDoc) Acc {field_m['accuracy']*100:.2f}% Macro-F1 {field_m['macro_f1']*100:.2f}%"
    )
    if combined > best_score:
        best_score = combined
        torch.save({
            "model_name": "convnext_tiny",
            "author": "khoi_domain_adapted",
            "num_classes": num_classes,
            "val_accuracy": lab_m["accuracy"],
            "val_f1_macro": lab_m["macro_f1"],
            "field_accuracy": field_m["accuracy"],
            "field_f1_macro": field_m["macro_f1"],
            "model_state_dict": model.state_dict(),
        }, SAVE_V2_PTH)
        print(f"   ⭐ Lưu Best Domain-Adapted Checkpoint -> {SAVE_V2_PTH}")

In [ ]:
# 4. Xuất ONNX FP32 + INT8 & chạy script đánh giá Domain Shift chính thức
if IN_COLAB:
    %pip install -q onnx onnxruntime

ckpt_v2 = torch.load(SAVE_V2_PTH, map_location="cpu", weights_only=False)
model_cpu = models.convnext_tiny(weights=None)
model_cpu.classifier[2] = nn.Linear(model_cpu.classifier[2].in_features, num_classes)
model_cpu.load_state_dict(ckpt_v2["model_state_dict"])
model_cpu.eval()

dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
torch.onnx.export(
    model_cpu, dummy, str(SAVE_V2_ONNX),
    export_params=True, opset_version=17, do_constant_folding=True,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
    dynamo=False,
)

from scripts.quantize_onnx import quantize_model
quantize_model(SAVE_V2_ONNX, SAVE_V2_INT8, mode="dynamic")

from scripts.eval_domain_shift import evaluate_on_plantdoc
evaluate_on_plantdoc(
    SAVE_V2_INT8,
    PLANTDOC_DIR / "test",
    PROJECT_DIR / "models" / "class_indices.json",
    OUT_REPORTS_DIR / "plantdoc_domain_shift_v2.json",
)